# Entregable Final: Assistent Immobiliari amb LLM i Guardrails

Aquest document conté la versió final del sistema de diàleg basat en LLM.
S'ha aplicat el concepte de Guardrails per evitar al·lucinacions, atacs de *Prompt Injection* i errors d'execució.

## 1. Càrrega de l'Entorn i el Model
A continuació inicialitzem les llibreries i carreguem el model LLM (Qwen2.5-Coder-3B-Instruct, que admet *tool calling*) a la memòria de la targeta gràfica (GPU).

In [1]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [2]:
import json
import re
from transformers import AutoModelForCausalLM, AutoTokenizer

# Carreguem el model amb suport per a "Function Calling" (Cridar Tools)
model_name = "Qwen/Qwen2.5-Coder-3B-Instruct"
print(f"Carregant el model {model_name} a la GPU...")

model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Model i Tokenizer carregats amb èxit!")

Carregant el model Qwen/Qwen2.5-Coder-3B-Instruct a la GPU...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

... (2 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model i Tokenizer carregats amb èxit!


## 2. Definició de la Tool (Motor de Cerca)
La funció de cerca `filter_real_estate` aplica **Tool Guardrails**:
1. S'inclouen filtres complexos (mínims i màxims, superfícies, booleans).
2. S'afegeix l'argument `**kwargs` per capturar paràmetres inventats pel model i retornar un error controlat (`UNSUPPORTED_FILTERS`) en lloc d'un `TypeError`.
3. Es converteixen tots els valors a número **una sola vegada** (`parse_num`) i les comparacions són segures davant de valors absents.
4. Si hi ha **més de 5 resultats** (`MAX_RESULTS`), la tool no els llista i demana a l'usuari que afineixi la cerca; així el bot mostra com a màxim 5 habitatges.
5. Un bloc `try/except` actua com a *Output Guardrail* i retorna errors controlats.

In [3]:
# ------------------------------------------------------------------
# 1. LÒGICA DE LA TOOL EN PYTHON
# ------------------------------------------------------------------
import json
import re

MAX_RESULTS = 5   # màxim d'habitatges que es poden llistar a l'usuari
CITIES = ["Barcelona", "L'Hospitalet de Llobregat", "Esplugues de Llobregat", "Santa Coloma de Gramenet"]

# Funcions auxiliars de neteja
def parse_num(val):
    """Converteix '250k', '300 mil', '1.200', 1200.0... a float. Retorna None si no es pot."""
    if val is None or isinstance(val, bool):
        return None
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).lower().strip().replace("€", "").replace("euros", "").replace("euro", "").strip()
    mult = 1
    if s.endswith("mil"):
        mult = 1000
        s = s[:-3].strip()
    elif s.endswith("k"):
        mult = 1000
        s = s[:-1].strip()
    # "1.200" o "1.200.000" -> separador de milers
    if re.fullmatch(r"\d{1,3}(\.\d{3})+", s):
        s = s.replace(".", "")
    s = s.replace(",", ".")
    try:
        return float(s) * mult
    except ValueError:
        return None

def normalize_type(t):
    if not t:
        return None
    text = str(t).lower().strip()
    if text in ["qualsevol", "any", "cualquiera", "indiferent"]:
        return "qualsevol"
    if text in ["buy", "comprar", "compra", "venda", "sale"]:
        return "sale"
    if text in ["rent", "llogar", "lloguer", "alquiler"]:
        return "rent"
    return text

def parse_bool(val):
    if val is None:
        return None
    if isinstance(val, bool):
        return val
    return str(val).lower().strip() in ["yes", "y", "true", "1", "sí", "si"]

def normalize_loc(loc):
    if not loc:
        return None
    text = str(loc).lower().strip()
    if text in ["qualsevol", "any", "cualquiera", "totes", "m'és igual", "es igual"]:
        return "qualsevol"
    if re.search(r"(hospitalet|hospi)", text): return "L'Hospitalet de Llobregat"
    if re.search(r"(esplugues|esplu)", text): return "Esplugues de Llobregat"
    if re.search(r"(coloma|santaco|gramenet)", text): return "Santa Coloma de Gramenet"
    if re.search(r"(barcelona|bcn|barna)", text): return "Barcelona"
    return text


def filter_real_estate(type=None, location=None, min_price=None, max_price=None,
                       min_bedrooms=None, min_bathrooms=None,
                       min_square_meters=None, max_square_meters=None,
                       floor=None, elevator=None, terrace=None, commercial_use=None,
                       income=None, **kwargs):
    """
    Filtra la base de dades. Accepta **kwargs per detectar paràmetres inventats sense petar.
    """
    try:
        if kwargs:
            return json.dumps({
                "status": "UNSUPPORTED_FILTERS",
                "system_instruction": f"No pots filtrar per aquestes característiques: {', '.join(kwargs.keys())}. Demana perdó a l'usuari i suggereix buscar sense aquests filtres."
            })

        with open('house_data.json', 'r', encoding='utf-8') as f:
            db = json.load(f)
        houses = db.get("houses", [])

        # 1. Comprovació de ciutat no suportada
        norm_loc_user = normalize_loc(location)
        if norm_loc_user and norm_loc_user != "qualsevol" and norm_loc_user not in CITIES:
            return json.dumps({
                "status": "UNSUPPORTED_CITY",
                "system_instruction": f"The city '{location}' is outside our operating area. Apologize and explicitly tell the user we ONLY work in Barcelona, L'Hospitalet, Esplugues, and Santa Coloma."
            })

        # 2. Normalitzem tots els paràmetres UNA sola vegada
        norm_type_user = normalize_type(type)
        p_min_price = parse_num(min_price)
        p_max_price = parse_num(max_price)
        p_income = parse_num(income)
        p_min_bed = parse_num(min_bedrooms)
        p_min_bath = parse_num(min_bathrooms)
        p_min_sqm = parse_num(min_square_meters)
        p_max_sqm = parse_num(max_square_meters)
        p_floor = parse_num(floor)

        # Límit efectiu de preu (en lloguer, el lloguer no pot superar el 35% dels ingressos)
        eff_max_price = p_max_price
        if norm_type_user == "rent" and p_income is not None:
            income_limit = p_income * 0.35
            if eff_max_price is None or income_limit < eff_max_price:
                eff_max_price = income_limit

        def matches_type_loc(h):
            if norm_type_user and norm_type_user != "qualsevol" and normalize_type(h.get("type")) != norm_type_user:
                return False
            if norm_loc_user and norm_loc_user != "qualsevol":
                norm_loc_db = normalize_loc(h.get("location"))
                if norm_loc_db and norm_loc_db != norm_loc_user:
                    return False
            return True

        def required(flag):
            return parse_bool(flag) is True

        # 3. Cerca estricta
        filtered_houses = []
        for h in houses:
            if not matches_type_loc(h):
                continue

            price = parse_num(h.get("price"))
            if price is not None:
                if p_min_price is not None and price < p_min_price: continue
                if eff_max_price is not None and price > eff_max_price: continue

            bedrooms = parse_num(h.get("bedrooms"))
            bathrooms = parse_num(h.get("bathrooms"))
            if p_min_bed is not None and bedrooms is not None and bedrooms < p_min_bed: continue
            if p_min_bath is not None and bathrooms is not None and bathrooms < p_min_bath: continue

            sqm = parse_num(h.get("square_meters"))
            if sqm is not None:
                if p_min_sqm is not None and sqm < p_min_sqm: continue
                if p_max_sqm is not None and sqm > p_max_sqm: continue

            h_floor = parse_num(h.get("floor"))
            if p_floor is not None and h_floor is not None and h_floor < p_floor: continue

            if required(elevator) and parse_bool(h.get("elevator")) is not True: continue
            if required(terrace) and parse_bool(h.get("terrace")) is not True: continue
            if required(commercial_use) and parse_bool(h.get("commercial_use")) is not True: continue

            filtered_houses.append(h)

        total = len(filtered_houses)

        # 4. Cerca relaxada (només si no hi ha cap resultat exacte)
        if total == 0:
            relaxed_houses = []
            for h in houses:
                if not matches_type_loc(h):
                    continue

                price = parse_num(h.get("price"))
                if eff_max_price is not None and price is not None and price > eff_max_price * 1.15:
                    continue

                bedrooms = parse_num(h.get("bedrooms"))
                if p_min_bed is not None and bedrooms is not None and bedrooms < p_min_bed - 1:
                    continue

                relaxed_houses.append(h)

            if len(relaxed_houses) > 0:
                return json.dumps({
                    "status": "PARTIAL_MATCHES",
                    "total_found": len(relaxed_houses),
                    "system_instruction": "No exact matches. Present these 2 as similar alternatives. Explicitly tell the user that they don't perfectly match, and the real reason why (e.g., slightly more expensive or missing an elevator/bedroom).",
                    "alternatives": relaxed_houses[:2]
                }, indent=2)
            else:
                return json.dumps({
                    "status": "ZERO_RESULTS",
                    "total_found": 0,
                    "system_instruction": "No properties found even with relaxed criteria. Suggest changing the city or drastically increasing the budget."
                })

        elif total > MAX_RESULTS:
            return json.dumps({
                "status": "TOO_MANY_RESULTS",
                "total_found": total,
                "system_instruction": f"You must NOT list the properties. Tell the user the total found (more than {MAX_RESULTS}), and ask them to filter by an unmentioned criterion (e.g. price, terrace, elevator)."
            })
        else:
            return json.dumps(filtered_houses, indent=2)

    except Exception as e:
        return json.dumps({"error": f"Error executant la cerca: {str(e)}"})


# ------------------------------------------------------------------
# 2. DEFINICIÓ JSON DE LA TOOL (El manual pel LLM)
# ------------------------------------------------------------------
tools = [
    {
        "type": "function",
        "function": {
            "name": "filter_real_estate",
            "description": "Searches a real estate database. Use ONLY defined parameters. Convert all string numbers like '250k' to integers. Only include a parameter if the user explicitly mentioned it.",
            "parameters": {
                "type": "object",
                "properties": {
                    "type": { "type": "string", "enum": ["rent", "sale", "qualsevol"], "description": "Transaction type. Use 'qualsevol' if user doesn't care." },
                    "location": { "type": "string", "enum": ["Barcelona", "L'Hospitalet de Llobregat", "Esplugues de Llobregat", "Santa Coloma de Gramenet", "qualsevol"], "description": "The exact city name. Use 'qualsevol' if user doesn't care." },
                    "min_price": { "type": "integer", "description": "Minimum price/budget in euros." },
                    "max_price": { "type": "integer", "description": "Maximum price/budget in euros." },
                    "income": { "type": "integer", "description": "Monthly income of the user in euros." },
                    "min_bedrooms": { "type": "integer", "description": "Minimum bedrooms required." },
                    "min_bathrooms": { "type": "integer", "description": "Minimum bathrooms required." },
                    "min_square_meters": { "type": "integer", "description": "Minimum size in sqm." },
                    "max_square_meters": { "type": "integer", "description": "Maximum size in sqm." },
                    "floor": { "type": "integer", "description": "Minimum floor level." },
                    "elevator": { "type": "string", "enum": ["Yes", "No"], "description": "Is an elevator strictly required?" },
                    "terrace": { "type": "string", "enum": ["Yes", "No"], "description": "Is a terrace strictly required?" },
                    "commercial_use": { "type": "string", "enum": ["Yes", "No"], "description": "Is commercial use required?" }
                },
                "required": ["type", "location"]
            }
        }
    }
]

AVAILABLE_TOOLS = {
    "filter_real_estate": filter_real_estate
}

# Paràmetres vàlids de la tool (s'usen per filtrar el que enviï el model)
VALID_PARAMS = set(tools[0]["function"]["parameters"]["properties"].keys())

print("Lògica de la base de dades i Esquema de Tools carregats!")

Lògica de la base de dades i Esquema de Tools carregats!


## 3. Input Guardrails, Output Guardrails i Motor d'Execució
S'ha programat un **Input Guardrail** basat en **Expressions Regulars (Regex)** que s'executa abans d'enviar l'input al model. Intercepta patrons de *Prompt Injection* (ex: "ignore instructions", "act as a pirate") i temes fora de domini, i bloqueja el missatge protegint el sistema.

També s'inclouen:
- **Guardrail de procedència de dades:** abans de guardar res a la memòria, es comprova que el `type` (comprar/llogar) i la `location` que ha enviat el model **els hagi dit realment l'usuari**. Si el model s'ho ha inventat, es descarta i es torna a preguntar a l'usuari.
- **Memòria global filtrada:** només es guarden els paràmetres vàlids de la tool. Els paràmetres inventats pel model s'apliquen només a la crida actual (perquè la tool avisi l'usuari) i no queden enganxats a la memòria.
- **Output Guardrail:** neteja la resposta final i, quan es llisten habitatges, força que el missatge acabi amb la frase de tancament establerta.
- Les funcions del patró **ReAct**, que reben la crida JSON, executen la Tool Python i re-injecten els resultats al context perquè el LLM en tregui conclusions.

In [4]:
# ------------------------------------------------------------------
# 3. INPUT GUARDRAILS
# ------------------------------------------------------------------
import re

def check_guardrails(text):
    """
    Analitza el text de l'usuari amb Expressions Regulars per detectar atacs o temes fora de domini.
    Retorna (True, text) si és segur, o (False, "Missatge d'error") si hi ha perill.
    """

    # 0. PROTECCIÓ DE TOKENS (Evita injeccions de rol simulant ser el sistema)
    special_tokens = [r"<\|im_start\|>", r"<\|im_end\|>", r"<\|endoftext\|>", r"```"]
    for token in special_tokens:
        if re.search(token, text):
            return False, "⚠️ [Seguretat Crítica]: S'ha detectat un intent d'injecció de format."

    text_lower = text.lower()

    # 1. JAILBREAK I CANVI DE ROL (Anglès, Català, Castellà)
    jailbreak_patterns = [
        # Anglès clàssic
        r"\b(ignore|forget).*(instructions)\b",
        r"\b(act\s+as|you\s+are\s+now)\b",
        r"\bsystem\s+prompt\b",
        r"\b(bypass|developer\s+mode)\b|\bdan\s+mode\b|\bdo\s+anything\s+now\b",
        # Català i Castellà
        r"\b(ignora|oblida|olvida|desestima).*(instruccions|instrucciones|regles|reglas|rules)\b",
        r"\b(actua com|fes de|ara ets|finge|actúa como|simula)\b",
        r"\b(revela|mostra|enséñame|dime).*(prompt|instruccions|instrucciones|configuració)\b"
    ]

    for pattern in jailbreak_patterns:
        if re.search(pattern, text_lower):
            return False, "⚠️ [Aturada de Seguretat]: Soc un assistent immobiliari i no puc canviar de rol ni ignorar instruccions internes."

    # 2. FORA DE DOMINI / OUT OF DOMAIN (OOD)
    ood_patterns = [
        # Tecnològic i Programació
        r"\b(python|java|javascript|html|css|c\+\+|sql|php|bash|react|angular|formula)\b",
        # Permetem "codi postal" usant un negative lookahead
        r"\b(codi|código)\b(?!\s+postal)",
        r"\b(code|script|programa|funció|función|algoritme|fibonacci|fibbonacci)\b",
        # Financer
        r"\b(crypto|bitcoin|ethereum|nft|bolsa|accions|trading)\b",
        # Altres àmbits quotidians o delicats
        # (NO bloquegem "cuina"/"cocina": és un terme habitual en immobiliària)
        r"\b(recepta|receptes|receta|recetas|poema|cançó|lletra|chiste|acudit|diagnòstic|metge|malaltia)\b"
    ]

    for pattern in ood_patterns:
        if re.search(pattern, text_lower):
            return False, "⚠️ [Fora de Domini]: Ho sento, però com a assistent immobiliari només et puc ajudar amb la compravenda o lloguer de pisos."

    # Si passa tots els filtres, endavant!
    return True, text


# ------------------------------------------------------------------
# 4. OUTPUT GUARDRAILS
# ------------------------------------------------------------------
CLOSING_SENTENCE = "Aquí tens les opcions disponibles."

def check_output_guardrails(response, last_tool_output=""):
    response_lower = response.lower()

    if re.search(r"\b(gratis|regal|regals|regalo|regala|regalar)\b", response_lower):
        return False, "⚠️ [Seguretat]: Ho sento, però com a política d'empresa no podem oferir pisos gratuïts o regals."

    if "```json" in response_lower or "\"name\": \"filter_real_estate\"" in response_lower:
        return False, "⚠️ [Error intern]: Disculpa, he tingut un petit error de format al processar la informació. M'ho pots tornar a demanar?"

    # 1. Filtre de males traduccions: només quan "Baixos" s'usa com a etiqueta (p. ex. "- Baixos: 2").
    #    No toquem "planta baixa" ni altres usos legítims de la paraula.
    response = re.sub(r"\b[Bb]aixos(\s*:)", r"Banys\1", response)

    # 2. Tisorada a les frases tòxiques (per evitar bucles al final de la conversa)
    response = response.replace("Si vols que es mostrin més propietats, simplement di-ho.", "")
    response = response.replace("Si vols fer una consulta més específica, també ho pots fer.", "")
    response = response.replace("Si t'agradaria més informació o si tens alguna altre pregunta, no dudes en preguntar.", "")

    # 3. Si s'ha llistat habitatges (cerca exacta), el missatge ha d'acabar amb la frase de tancament
    #    i sense cap pregunta o suggeriment addicional.
    if "**Propietat" in response and "PARTIAL_MATCHES" not in last_tool_output:
        localitats = list(re.finditer(r"Localitat:[^\n]*", response))
        if localitats:
            response = response[:localitats[-1].end()].rstrip() + "\n\n" + CLOSING_SENTENCE

    return True, response.strip()


# ------------------------------------------------------------------
# 5. GUARDRAIL DE PROCEDÈNCIA DE DADES (el model no pot inventar type/location)
# ------------------------------------------------------------------
BUY_RE  = re.compile(r"\b(compr\w*|cmpr\w*|vend\w*|buy\w*|purchas\w*|sale)\b")
RENT_RE = re.compile(r"\b(llog\w*|lloguer\w*|alquil\w*|arrend\w*|rent\w*)\b")
ANY_RE  = re.compile(r"(qualsevol|m'?és igual|m'?es igual|tant és|tant da|me da igual|cualquiera|indiferent|\bany\b)")

CITY_ALIASES = {
    "Barcelona": r"barcelona|bcn|barna",
    "L'Hospitalet de Llobregat": r"hospitalet|hospi",
    "Esplugues de Llobregat": r"esplugues|esplu",
    "Santa Coloma de Gramenet": r"coloma|santaco|gramenet",
}

def get_user_text(messages):
    """Text real escrit per l'usuari (ignora els missatges de sistema injectats com a 'user')."""
    parts = []
    for m in messages:
        if m.get("role") != "user":
            continue
        c = m.get("content") or ""
        if c.startswith(("System Directive", "System Error", "Error:", "You already executed")):
            continue
        parts.append(c.lower())
    return " ".join(parts)

def validate_arguments(arguments, user_text):
    """
    Comprova que 'type' i 'location' enviats pel model apareguin realment al que ha dit l'usuari.
    Retorna (arguments_nets, llista_de_rebutjats).
    """
    clean, rejected = dict(arguments), []

    # TYPE
    t = clean.get("type")
    if t not in (None, "", "ESBORRAR"):
        nt = normalize_type(t)
        ok = ((nt == "sale" and BUY_RE.search(user_text)) or
              (nt == "rent" and RENT_RE.search(user_text)) or
              (nt == "qualsevol" and ANY_RE.search(user_text)))
        if not ok:
            rejected.append("type")
            clean.pop("type")

    # LOCATION
    loc = clean.get("location")
    if loc not in (None, "", "ESBORRAR"):
        nl = normalize_loc(loc)
        if nl == "qualsevol":
            ok = bool(ANY_RE.search(user_text))
        elif nl in CITY_ALIASES:
            ok = bool(re.search(CITY_ALIASES[nl], user_text))
        else:
            # Ciutat no suportada: només la deixem passar si l'usuari l'ha dita
            # (així la tool la rebutja amb UNSUPPORTED_CITY).
            ok = str(loc).lower().split()[0] in user_text
        if not ok:
            rejected.append("location")
            clean.pop("location")

    return clean, rejected


# ------------------------------------------------------------------
# 6. MEMÒRIA GLOBAL
# ------------------------------------------------------------------
memoria_global = {k: None for k in VALID_PARAMS}

def reset_memory():
    for k in VALID_PARAMS:
        memoria_global[k] = None


# ------------------------------------------------------------------
# 7. MOTOR REACT
# ------------------------------------------------------------------
def call_tool(messages, function_name, arguments, clean_assistant_content, verbose=True):
    if verbose:
        print(f"\n[🔧 Executing Tool]: {function_name} with parameters: {arguments}")

    if function_name in AVAILABLE_TOOLS:
        tool_output = AVAILABLE_TOOLS[function_name](**arguments)

        if tool_output == "[]" or not tool_output:
            tool_output = "No houses matching all filtering criteria were found in the database."
    else:
        tool_output = json.dumps({"error": f"Unknown tool '{function_name}'. Use only filter_real_estate."})

    if verbose:
        print(f"[📥 Tool Output Payload]: {tool_output}")

    messages.append({"role": "assistant", "content": clean_assistant_content})
    messages.append({"role": "tool", "name": str(function_name), "content": str(tool_output)})
    return messages

def query_model_with_tools(messages, verbose=True, current_iteration=1, max_iterations=5):
    global memoria_global

    if current_iteration > max_iterations:
        return "Disculpa, no he pogut completar la tasca en el temps esperat."

    text = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    im_end_id = tokenizer.encode("<|im_end|>")[0]

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=False,
        eos_token_id=[im_end_id, tokenizer.eos_token_id],
        pad_token_id=tokenizer.eos_token_id
    )

    new_tokens = generated_ids[0][len(model_inputs.input_ids[0]):]
    decoded_message = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    raw_message = tokenizer.decode(new_tokens, skip_special_tokens=False).strip()
    clean_assistant_content = raw_message.replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()

    json_block_match = re.search(r"```json\s*(.*?)\s*```", decoded_message, re.DOTALL)

    if json_block_match:
        tool_call_payload = json_block_match.group(1)

        # Parseig del JSON (únic punt on pot fallar el format)
        try:
            tool_call_json = json.loads(tool_call_payload)
            if not isinstance(tool_call_json, dict):
                raise json.JSONDecodeError("El JSON no és un objecte", tool_call_payload, 0)
        except json.JSONDecodeError:
            messages.append({"role": "assistant", "content": clean_assistant_content})
            messages.append({"role": "user", "content": "Error: Your output is not valid JSON. Please format it with ```json."})
            return query_model_with_tools(messages, verbose, current_iteration + 1, max_iterations)

        function_name = tool_call_json.get("name")
        arguments = tool_call_json.get("arguments", {})
        if not isinstance(arguments, dict):
            arguments = {}

        # ---> 1. GUARDRAIL DE PROCEDÈNCIA: type/location han de venir de l'usuari <---
        arguments, rejected = validate_arguments(arguments, get_user_text(messages))
        if verbose and rejected:
            print(f"🛡️ [Guardrail Intern]: El model ha inventat {rejected}. Descartat.")

        # ---> 2. SEPAREM PARÀMETRES VÀLIDS I INVENTATS <---
        known_args = {k: v for k, v in arguments.items() if k in VALID_PARAMS}
        extra_args = {k: v for k, v in arguments.items() if k not in VALID_PARAMS}

        # ---> 3. ACTUALITZACIÓ DE LA MEMÒRIA (només paràmetres vàlids) <---
        for clau, valor in known_args.items():
            valor_net = str(valor).strip()

            # Si l'usuari demana explícitament esborrar el filtre, el posem a None
            if valor_net == "ESBORRAR":
                memoria_global[clau] = None

            # Si l'LLM ens envia una dada nova vàlida, l'actualitzem.
            # Si ens envia un "null"/buit, l'IGNOREM per protegir el que ja sabíem d'abans.
            elif valor is not None and valor_net != "" and valor_net.lower() not in ("none", "null"):
                memoria_global[clau] = valor

        # ---> 4. GUARDRAIL BASAT EN LA MEMÒRIA GLOBAL <---
        falta = []
        if not memoria_global.get("type"):
            falta.append("si vol comprar o llogar")
        if not memoria_global.get("location"):
            falta.append("a quina ciutat")

        has_budget = False
        for k in ["max_price", "min_price", "min_bedrooms", "income"]:
            if memoria_global.get(k) is not None and str(memoria_global.get(k)).strip() != "":
                has_budget = True
                break

        if not has_budget:
            falta.append("el pressupost, sou o habitacions")

        # Si falten els requisits mínims, aturem i demanem
        if falta:
            directive = f"System Directive: Do not execute the tool yet. You are missing: {', '.join(falta)}. Ask the user EXACTLY for this missing information."
            if extra_args:
                directive += f" Also apologize because we cannot filter by: {', '.join(extra_args.keys())}."
            temp_messages = messages.copy()
            temp_messages.append({"role": "assistant", "content": clean_assistant_content})
            temp_messages.append({"role": "user", "content": directive})
            return query_model_with_tools(temp_messages, verbose, current_iteration + 1, max_iterations)

        # ---> 5. CONTROL ANTI-BUCLES I EXECUCIÓ DE LA TOOL <---
        prev_content = messages[-2].get("content") or "" if len(messages) >= 2 and messages[-2].get("role") == "assistant" else ""
        if function_name and function_name in prev_content:
            messages.append({
                "role": "user",
                "content": "You already executed that tool. Please answer the user directly."
            })
        else:
            # Els paràmetres inventats només viatgen en aquesta crida (la tool avisarà l'usuari)
            messages = call_tool(messages, function_name, {**memoria_global, **extra_args}, clean_assistant_content, verbose)
        return query_model_with_tools(messages, verbose, current_iteration + 1, max_iterations)

    # ---> GUARDRAIL ANTI-REPETICIÓ (AUTOCORRECCIÓ INVISIBLE) <---
    if len(messages) >= 2 and messages[-2].get("role") == "assistant":
        if clean_assistant_content.strip() == (messages[-2].get("content") or "").strip():
            if verbose:
                print("🛡️ [Guardrail Intern]: El model ha entrat en bucle de text. Obligant-lo a regenerar...")

            messages.append({"role": "assistant", "content": clean_assistant_content})
            messages.append({
                "role": "user",
                "content": "System Directive: You just repeated your exact previous message. You MUST read my latest input carefully and provide a DIFFERENT, helpful response."
            })
            return query_model_with_tools(messages, verbose, current_iteration + 1, max_iterations)

    return clean_assistant_content

print("Guardrails i Funcions ReAct carregats!")

Guardrails i Funcions ReAct carregats!


## 4. System Prompt i Inicialització
Finalment, apliquem diverses tècniques de prompting com **Few-Shot Prompting** (afegint exemples exactes de com volem que retorni el JSON) per reduir les al·lucinacions de format encara que l'usuari parli en llengües minoritàries com el català.

El bot mostra **com a màxim 5 habitatges** per cerca: si n'hi ha més, la tool ho indica i el bot demana a l'usuari que afineixi els criteris. Quan es llisten habitatges, el missatge acaba sempre amb la frase *"Aquí tens les opcions disponibles."*

In [ ]:
# ------------------------------------------------------------------
# 8. SYSTEM PROMPT I INICIALITZACIÓ
# ------------------------------------------------------------------

session_history = [
    {
        "role": "system",
        "content": (
            "You are an expert, professional real estate assistant.\n\n"

            "--- CORE BEHAVIOR (JSON FIRST) ---\n"
            "1. EXTRACT EVERYTHING CAREFULLY: Read the ENTIRE user input carefully from start to finish. You MUST extract ALL real estate parameters mentioned (type, location, budget, bedrooms, etc.) and output them together in a single ```json block BEFORE saying anything else. NEVER ignore or drop a parameter the user has provided, to avoid asking them for information they have already answered.\n"
            "2. CONVERSATION: After your ```json block, read the hidden 'System Directive' (if any) and politely ask the user for the missing information it indicates. Only ask for missing information.\n"
            "3. NUMBER CONVERSION: Convert 'k' to thousands (300k = 300000). You MUST also convert numbers written in words into digits (e.g., 'cinc-cents mil' = 500000, 'dos mil' = 2000).\n"
            "4. CLEARING FILTERS: If the user says they don't care about a filter anymore, output \"ESBORRAR\".\n\n"

            "--- STRICT RULES ---\n"
            "5. STRICT CATCH-ALL & OFF-TOPIC: You are strictly a real estate search engine. NEVER engage in general conversation. If the user talks about ANYTHING unrelated to finding a house (e.g. weather, advice, personal questions, general knowledge), you MUST reply EXACTLY with: 'Com a assistent immobiliari, el meu únic propòsit és ajudar-te a buscar pisos amb la nostra base de dades. Què necessites, comprar o llogar?'\n"
            "6. DATABASE LIMIT: Our database ONLY has houses in 'Barcelona', 'L'Hospitalet de Llobregat', 'Esplugues de Llobregat', and 'Santa Coloma de Gramenet'. If the user asks for ANY other city, apologize and explain we only operate in those 4 cities.\n"
            "7. NO UNKNOWN INFO & NO HALLUCINATIONS: NEVER invent properties, prices, or extra features. If the user asks ANY question about a property or a feature that cannot be answered using the exact fields in our database (e.g. asking if it's bright, if it allows pets, about the neighbors, the neighborhood quality, etc.), you MUST explicitly decline by saying: 'Ho sento, només et puc donar la informació bàsica de la base de dades i no puc respondre aquesta pregunta. Continuem amb la cerca?'\n"
            "8. UNSUPPORTED FILTERS CATCH-ALL: You can ONLY filter by: type, location, min_price, max_price, bedrooms, bathrooms, square_meters, floor, elevator, terrace, commercial_use, income. If the user asks for ABSOLUTELY ANY OTHER FEATURE (e.g. 'microones', 'balcó', 'jardí', 'garatge', 'piscina', 'mascotes', etc.), you MUST prioritize detecting it. You must output the valid JSON, but you MUST explicitly apologize in your text saying we cannot filter by that specific feature.\n"            
            "9. ANY PARAMETER: If the user says 'any', use \"qualsevol\". For Budget use 9999999. NEVER say '9999999' out loud to the user.\n\n"
            "10. NO QUESTIONS AT THE END: When you list the properties found, end your message exactly with 'Aquí tens les opcions disponibles.' DO NOT ask 'Quina opció prefereixes?' or any other questions.\n"

            "--- FEW-SHOT EXAMPLES ---\n\n"

            "EXAMPLE 1 (CONVERSATION FLOW WITH SYSTEM DIRECTIVES):\n"
            "User: \"Vull llogar\"\n"
            "Assistant: ```json\n"
            "{\n  \"name\": \"filter_real_estate\",\n  \"arguments\": { \"type\": \"rent\" }\n}\n```\n"
            "User: System Directive: Do not execute the tool yet. You are missing: a quina ciutat, el pressupost, sou o habitacions. Ask the user EXACTLY for this missing information.\n"
            "Assistant: D'acord, buscarem opcions de lloguer! A quina ciutat t'interessa buscar, i quin és el teu pressupost màxim?\n"
            "User: \"A Barcelona\"\n"
            "Assistant: ```json\n"
            "{\n  \"name\": \"filter_real_estate\",\n  \"arguments\": { \"location\": \"Barcelona\" }\n}\n```\n"
            "User: System Directive: Do not execute the tool yet. You are missing: el pressupost, sou o habitacions. Ask the user EXACTLY for this missing information.\n"
            "Assistant: Perfecte, buscarem a Barcelona. Per poder acotar la cerca, quin és el teu pressupost màxim?\n"
            "User: \"uns 300k\"\n"
            "Assistant: ```json\n"
            "{\n  \"name\": \"filter_real_estate\",\n  \"arguments\": { \"max_price\": 300000 }\n}\n```\n"
            "User: Tool returned: [{\"type\": \"rent\", \"price\": 1200, \"location\": \"Barcelona\"}]\n"
            "Assistant: Aquí tens les opcions disponibles.\n\n"

            "EXAMPLE 2 (EXTRACT MULTIPLE PARAMETERS):\n"
            "User: \"a Barcelona, preu màxim 300k\"\n"
            "Assistant: ```json\n"
            "{\n  \"name\": \"filter_real_estate\",\n  \"arguments\": { \"location\": \"Barcelona\", \"max_price\": 300000 }\n}\n```\n\n"

            "EXAMPLE 3 (EXTRACT JUST A NUMBER):\n"
            "User: \"300000\"\n"
            "Assistant: ```json\n"
            "{\n  \"name\": \"filter_real_estate\",\n  \"arguments\": { \"max_price\": 300000 }\n}\n```\n\n"

            "EXAMPLE 4 (EXTRACT ALL PARAMS, PARSE WORDS, AND APOLOGIZE FOR UNSUPPORTED FILTERS):\n"
            "User: \"Vull comprar a Barcelona amb microones, fins a cinc-cents mil euros\"\n"
            "Assistant: ```json\n"
            "{\n  \"name\": \"filter_real_estate\",\n  \"arguments\": { \"type\": \"sale\", \"location\": \"Barcelona\", \"max_price\": 500000 }\n}\n```\n"
            "User: Tool returned: [{\"type\": \"sale\", \"price\": 450000, \"bedrooms\": 2, \"bathrooms\": 1, \"square_meters\": 75, \"floor\": 2, \"elevator\": \"Sí\", \"commercial_use\": \"No\", \"terrace\": \"No\", \"location\": \"Barcelona\"}]\n"
            "Assistant: Ho sento, no tenim el filtre de microones a la base de dades. Tot i així, he aplicat els altres paràmetres.\n"

            "EXAMPLE 5 (UNSUPPORTED FILTERS):\n"
            "User: \"Vull un pis amb garatge\"\n"
            "Assistant: Ho sento, no tenim aquest filtre a la base de dades.\n"

            "EXAMPLE 6 (UNSUPPORTED FILTER AT START):\n"
            "User: \"A alguna de les propietats puc portar el meu gos?\"\n"
            "Assistant: ```json\n"
            "{\n  \"name\": \"filter_real_estate\",\n  \"arguments\": { }\n}\n```\n"
            "User: System Directive: Do not execute the tool yet. You are missing: si vol comprar o llogar, a quina ciutat, el pressupost, sou o habitacions. Ask the user EXACTLY for this missing information.\n"
            "Assistant: Ho sento, el nostre sistema no té informació sobre mascotes o gossos. Com que hem tancat la cerca anterior, què necessites per a una nova cerca: comprar o llogar?\n\n"

            "EXAMPLE 7 (UNSUPPORTED CITY):\n"
            "User: \"Vull llogar a Salou\"\n"
            "Assistant: ```json\n"
            "{\n  \"name\": \"filter_real_estate\",\n  \"arguments\": { \"type\": \"rent\" }\n}\n```\n"
            "Ho sento, però actualment només treballem a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. T'interessa buscar en alguna d'aquestes quatre ciutats?\n\n"

            "EXAMPLE 8 (ZERO RESULTS JSON):\n"
            "User: Tool returned: {\"status\": \"ZERO_RESULTS\", \"total_found\": 0}\n"
            "Assistant: Ho sento molt, però ara mateix no tenim cap pis amb aquestes característiques exactes. Et semblaria bé provar amb una altra ciutat o ajustar una mica el pressupost?\n\n"

            "EXAMPLE 9 (ZERO RESULTS TEXT - CALL TOOL FALLBACK):\n"
            "User: Tool returned: No houses matching all filtering criteria were found in the database.\n"
            "Assistant: Ho sento molt, però ara mateix no tenim cap pis amb aquestes característiques a la nostra base de dades. Et semblaria bé provar amb una altra ciutat o ajustar el pressupost?\n\n"

            "EXAMPLE 10 (SUCCESSFUL SEARCH RESULTS):\n"
            "User: Tool returned: [{\"type\": \"sale\", \"price\": 220000, \"bedrooms\": 3, \"bathrooms\": 2, \"square_meters\": 95, \"floor\": 4, \"elevator\": \"Sí\", \"commercial_use\": \"No\", \"terrace\": \"Sí\", \"location\": \"Barcelona\"}]\n"
            "Assistant: Aquí tens les opcions disponibles:\n\n"
            "1. **Propietat 1**\n"
            "  - Tipus: Venda\n"
            "  - Habitacions: 3\n"
            "  - Banys: 2\n"
            "  - Preu: 220.000€\n"
            "  - Metres quadrats: 95\n"
            "  - Planta: 4\n"
            "  - Ascensor: Sí\n"
            "  - Ús comercial: No\n"
            "  - Terrassa: Sí\n"
            "  - Localitat: Barcelona\n\n"
            "Aquí tens les opcions disponibles.\n\n"

            )
    }
]

def start_chat_session_segur(history):
    # Reiniciem memòria i historial (només es conserva el System Prompt),
    # així es pot tornar a executar la cel·la sense acumular missatges.
    reset_memory()
    del history[1:]

    TOOL_STATUS_NO_CLOSE = ["ZERO_RESULTS", "TOO_MANY_RESULTS", "UNSUPPORTED_CITY", "UNSUPPORTED_FILTERS", "error"]

    print("====================================================")
    print("🏠 Assistent Immobiliari Segur Inicialitzat!")
    print("ℹ️ Escriu 'sortir' per tancar la conversa.")
    print("====================================================")

    # PRENEM LA INICIATIVA
    salutacio_inicial = "Hola! Soc el teu assistent immobiliari. Treballem exclusivament a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. Per començar, què busques: comprar o llogar?"

    # L'afegim a l'historial perquè el model sàpiga que ho ha dit ell
    history.append({"role": "assistant", "content": salutacio_inicial})
    print(f"\nAssistent: {salutacio_inicial}")

    while True:
        try:
            user_input = input("\nTu: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nGràcies per utilitzar l'assistent. Fins aviat!")
            break

        if user_input.lower() in ['exit', 'quit', 'sortir', 'adeu']:
            print("Gràcies per utilitzar l'assistent. Fins aviat!")
            reset_memory()
            break

        if not user_input:
            continue

        # --> INTERVENCIÓ DEL GUARDRAIL D'ENTRADA <--
        is_safe, warning_msg = check_guardrails(user_input)
        if not is_safe:
            print(f"\nAssistent: {warning_msg}")
            continue

        # Si és segur, continuem amb el flux de l'agent
        history.append({"role": "user", "content": user_input})
        final_response = query_model_with_tools(history, verbose=False)

        # Última sortida de la tool (si aquest torn n'ha cridat una)
        last_tool = history[-1].get("content", "") if history[-1]["role"] == "tool" else ""

        # --> INTERVENCIÓ DE L'OUTPUT GUARDRAIL <--
        out_safe, safe_response = check_output_guardrails(final_response, last_tool)

        if history[-1]["role"] != "assistant":
            history.append({"role": "assistant", "content": safe_response})

        print(f"\nAssistent: {safe_response}")

        # ---> Tancament intel·ligent <---
        # Només si la tool ha retornat una llista real d'habitatges i el bot l'ha mostrada
        tool_ok = bool(last_tool) and not any(s in last_tool for s in TOOL_STATUS_NO_CLOSE)
        if tool_ok and "**Propietat" in safe_response and "?" not in safe_response:

            print("\n[Sistema]: 🏠 Cerca finalitzada amb èxit.")
            print("Assistent: Vols buscar un altre habitatge? (Escriu què busques per fer una nova cerca o escriu 'sortir' per tancar)")

            # 1. Buidem la memòria global perquè la nova cerca comenci en blanc
            reset_memory()

            # 2. Buidem l'historial de l'LLM (conservant només el System Prompt)
            del history[1:]

In [6]:
# CERCA DIRECTA / BORRAR MEMÒRIA (NOVA CERCA) / CERCA PER PASSOS
start_chat_session_segur(session_history)

🏠 Assistent Immobiliari Segur Inicialitzat!
ℹ️ Escriu 'sortir' per tancar la conversa.

Assistent: Hola! Soc el teu assistent immobiliari. Treballem exclusivament a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. Per començar, què busques: comprar o llogar?

Tu: Vull comprar a Barcelona fins a 300k

Assistent: Aquí tens les opcions disponibles:

1. **Propietat 1**
  - ID: 23
  - Tipus: Venda
  - Habitacions: 3
  - Banys: 2
  - Preu: 285.000€
  - Metres quadrats: 95
  - Planta: 1
  - Ascensor: Sí
  - Ús comercial: No
  - Terrassa: No
  - Localitat: Barcelona

2. **Propietat 2**
  - ID: 24
  - Tipus: Venda
  - Habitacions: 1
  - Banys: 1
  - Preu: 200.000€
  - Metres quadrats: 50
  - Planta: 2
  - Ascensor: No
  - Ús comercial: No
  - Terrassa: Sí
  - Localitat: Barcelona

3. **Propietat 3**
  - ID: 25
  - Tipus: Venda
  - Habitacions: 2
  - Banys: 1
  - Preu: 180.000€
  - Metres quadrats: 60
  - Planta: 3
  - Ascensor: No
  - Ús comercial: Sí
  - Terrassa: No
  - Localitat: Barcelona

In [8]:
# CONVERSA LLARGA, MOLTES CARACTERÍSTIQUES, SAP PER QUÈ POT FILTRAR, QUALSEVOL
start_chat_session_segur(session_history)

🏠 Assistent Immobiliari Segur Inicialitzat!
ℹ️ Escriu 'sortir' per tancar la conversa.

Assistent: Hola! Soc el teu assistent immobiliari. Treballem exclusivament a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. Per començar, què busques: comprar o llogar?

Tu: vull comprar a qualsevol ciutat

Assistent: D'acord, buscarem opcions de venda! Quin és el teu pressupost màxim?

Tu: 1500000

Assistent: Ho sento, però ara mateix tenim més de 5 propietats que compleixen les teves condicions. Pots filtrar per un altre criteri, com el preu, la terrassa, l'ascensor, o qualsevol altre que no hagis mencionat?

Tu: que tingui terrassa

Assistent: Ho sento, però ara mateix tenim més de 5 propietats que compleixen les teves condicions. Pots filtrar per un altre criteri, com el preu, la terrassa, l'ascensor, o qualsevol altre que no hagis mencionat?

Tu: per quins criteris pots filtrar?

Assistent: Pots filtrar per preu, terrassa, ascensor, habitacions, banys, metres quadrats, planta, us comercial,

In [16]:
# 35% INGRESSOS / CANVI D'INFORMACIÓ / BARNA
start_chat_session_segur(session_history)

🏠 Assistent Immobiliari Segur Inicialitzat!
ℹ️ Escriu 'sortir' per tancar la conversa.

Assistent: Hola! Soc el teu assistent immobiliari. Treballem exclusivament a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. Per començar, què busques: comprar o llogar?

Tu: vull llogar un pis a barna

Assistent: D'acord, buscarem opcions de lloguer! Quin és el teu pressupost màxim?

Tu: ai no! vull llogar a l'hospitalet

Assistent: Perfecte, buscarem a L'Hospitalet. Per poder acotar la cerca, quin és el teu pressupost màxim?

Tu: cobro 2000 euros al mes

Assistent: Ho sento molt, però ara mateix no tenim cap pis amb aquestes característiques a la nostra base de dades. Et semblaria bé provar amb una altra ciutat o ajustar el pressupost?

Tu: doncs a esplugues

Assistent: Aquí tens les opcions disponibles:

1. **Propietat 1**
  - Tipus: Lloguer
  - Habitacions: 1
  - Banys: 1
  - Preu: 700€
  - Metres quadrats: 55
  - Planta: 3
  - Ascensor: No
  - Ús comercial: No
  - Terrassa: Sí
  - Localitat:

In [10]:
# MOLTS FILTRES
start_chat_session_segur(session_history)

🏠 Assistent Immobiliari Segur Inicialitzat!
ℹ️ Escriu 'sortir' per tancar la conversa.

Assistent: Hola! Soc el teu assistent immobiliari. Treballem exclusivament a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. Per començar, què busques: comprar o llogar?

Tu: Vull comprar a Esplugues, 3 habitacions, amb terrassa i ascensor, fins a 400k

Assistent: Aquí tens les opcions disponibles:

1. **Propietat 1**
  - ID: 8
  - Tipus: Venda
  - Habitacions: 4
  - Banys: 2
  - Preu: 400.000€
  - Metres quadrats: 150
  - Planta: 5
  - Ascensor: Sí
  - Ús comercial: Sí
  - Terrassa: Sí
  - Localitat: Esplugues de Llobregat

2. **Propietat 2**
  - ID: 12
  - Tipus: Venda
  - Habitacions: 4
  - Banys: 2
  - Preu: 350.000€
  - Metres quadrats: 100
  - Planta: 6
  - Ascensor: Sí
  - Ús comercial: Sí
  - Terrassa: Sí
  - Localitat: Esplugues de Llobregat

Aquí tens les opcions disponibles.

[Sistema]: 🏠 Cerca finalitzada amb èxit.
Assistent: Vols buscar un altre habitatge? (Escriu què busques per fer

In [15]:
# CIUTAT FORA D'ABAST / RESULTATS PARCIALS
start_chat_session_segur(session_history)

🏠 Assistent Immobiliari Segur Inicialitzat!
ℹ️ Escriu 'sortir' per tancar la conversa.

Assistent: Hola! Soc el teu assistent immobiliari. Treballem exclusivament a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. Per començar, què busques: comprar o llogar?

Tu: vull un pis a Madrid

Assistent: Ho sento, però actualment només treballem a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. T'interessa buscar en alguna d'aquestes quatre ciutats?

Tu: Doncs comprar a barcelona. 5 habitacions i pressupost 500k 

Assistent: Ho sento, però ara mateix no tenim cap pis amb 5 habitacions a Barcelona que compleixi tots els teus requisits exactes. T'he presentat dues alternatives similars, però amb diferències significatives:

1. **Propietat 1**
  - Tipus: Venda
  - Habitacions: 4
  - Banys: 2
  - Preu: 480.000€
  - Metres quadrats: 120
  - Planta: 3
  - Ascensor: Sí
  - Ús comercial: Sí
  - Terrassa: Sí
  - Localitat: Barcelona

2. **Propietat 2**
  - Tipus: Venda
  - Habitacions: 4
  - Banys:

In [23]:
# NO ALUCINA (FILTRES NO SUPORTATS)
start_chat_session_segur(session_history)

🏠 Assistent Immobiliari Segur Inicialitzat!
ℹ️ Escriu 'sortir' per tancar la conversa.

Assistent: Hola! Soc el teu assistent immobiliari. Treballem exclusivament a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. Per començar, què busques: comprar o llogar?

Tu: comprar a barcelona

Assistent: D'acord, buscarem opcions de venda! Quin és el teu pressupost màxim?

Tu: vull saber si puc portar el meu gos

Assistent: Ho sento, no tenim el filtre de mascotes a la base de dades. Tot i així, he aplicat els altres paràmetres.

Tu: vull saber si hi ha boscos

Assistent: Ho sento, no tenim el filtre de boscos a la base de dades. Tot i així, he aplicat els altres paràmetres.

Tu: sortir
Gràcies per utilitzar l'assistent. Fins aviat!


In [24]:
# INFORMACIÓ DESCONEGUDA / OFF TOPIC / BARNA
start_chat_session_segur(session_history)

🏠 Assistent Immobiliari Segur Inicialitzat!
ℹ️ Escriu 'sortir' per tancar la conversa.

Assistent: Hola! Soc el teu assistent immobiliari. Treballem exclusivament a Barcelona, L'Hospitalet, Esplugues i Santa Coloma. Per començar, què busques: comprar o llogar?

Tu: donam la formula de lecuacio de segon grau

Assistent: ⚠️ [Fora de Domini]: Ho sento, però com a assistent immobiliari només et puc ajudar amb la compravenda o lloguer de pisos.

Tu: vull que mexpliquis un conte

Assistent: Com a assistent immobiliari, el meu únic propòsit és ajudar-te a buscar pisos amb la nostra base de dades. Què necessites, comprar o llogar?

Tu: llogar un pis a barna per 800 euros

Assistent: Aquí tens les opcions disponibles:

1. **Propietat 1**
  - Tipus: Lloguer
  - Habitacions: 1
  - Banys: 1
  - Preu: 450€
  - Metres quadrats: 45
  - Planta: 6
  - Ascensor: No
  - Ús comercial: No
  - Terrassa: No
  - Localitat: Barcelona

2. **Propietat 2**
  - Tipus: Lloguer
  - Habitacions: 3
  - Banys: 1
  - Pr